# 01_train_final

- Full training run of DeclipNet using winning loss function from ablation study: weighted_l1 + scaled DWT
- Same architecture as ablation study (H=8, N_ATTN=3, NUM_HEADS=4, FFN_DIM=256)
- Stricter early stopping than ablation study to ensure full convergence
- Checkpoints best model by validation SI-SDR

## Limitations and future work
- Hyperparameter tuning (learning rate, batch size, model width H, attention heads,
  number of attention blocks) was not undertaken due to the computational cost of
  training on local MPS hardware. Default values were used throughout.
- Loss function ablation runs were relatively short and subject to variance
- A learning rate schedule (reduce on plateau) applied
- Model capacity (H=8, ~314k parameters) is intentionally conservative for local training;
  scaling H would likely improve performance at the cost of training time
- These are natural directions for future work given access to more compute

In [1]:
# Imports

import sys
import subprocess
import torch
import torch.optim as optim
from torch.utils.data import DataLoader

sys.path.insert(0, "..")
from config import *
from util import DeclipNet, DeclipDataset, make_loss_fn, train_run, si_sdr

In [2]:
# Hyperparameters
BATCH_SIZE = 32
LR = 1e-3
PATIENCE = 20        # epochs without improvement before early stopping (x val_every effective epochs)
MIN_DELTA = 1e-3     # minimum SI-SDR improvement to count as progress
VAL_EVERY = 1
GRAD_CLIP = None

# Model
H = 8
N_ATTN = 3
NUM_HEADS = 4
FFN_DIM = 256

# Device
DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Device: {DEVICE}")

Device: mps


In [3]:
# Dataset and dataloaders
train_dataset = DeclipDataset(TRAIN_OUT / "train_blocks.pt", TRAIN_OUT / "train_manifest.json")
val_dataset = DeclipDataset(VAL_OUT / "val_blocks.pt", VAL_OUT / "val_manifest.json")

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples:   {len(val_dataset)}")

Train samples: 225045
Val samples:   28131


## Training optimizations for local MPS hardware

**Candidates**
- Val every X epochs — 1/X validation overhead (28k samples per eval)
- Gradient clipping — prevents destabilizing large updates
- torch.compile — JIT compilation, MPS support partial, remove if errors

In [4]:
# Training run
config = {"name": "weighted_l1_dwt"}

caffeinate = subprocess.Popen(["caffeinate", "-i"])
try:
    train_run(
        run_name=config["name"],
        loss_fn=make_loss_fn(config["name"]),
        train_loader=train_loader,
        val_loader=val_loader,
        device=DEVICE,
        study_out=FINAL_OUT,
        lr=LR,
        patience=PATIENCE,
        min_delta=MIN_DELTA,
        lr_scheduling=True,
        val_every=VAL_EVERY,
        compile_model=True,
        grad_clip=GRAD_CLIP
    )
finally:
    caffeinate.terminate()
    print("Caffeinate terminated.")

[weighted_l1_dwt] epoch 001 | loss: 0.227175 | val SI-SDR: 15.5355 dB | best: 15.5355 dB | lr: 1.00e-03 | no improve: 0/20 (0/20 epochs)
[weighted_l1_dwt] epoch 002 | loss: 0.158304 | val SI-SDR: 16.9259 dB | best: 16.9259 dB | lr: 1.00e-03 | no improve: 0/20 (0/20 epochs)
[weighted_l1_dwt] epoch 003 | loss: 0.142359 | val SI-SDR: 18.0134 dB | best: 18.0134 dB | lr: 1.00e-03 | no improve: 0/20 (0/20 epochs)
[weighted_l1_dwt] epoch 004 | loss: 0.133043 | val SI-SDR: 18.6417 dB | best: 18.6417 dB | lr: 1.00e-03 | no improve: 0/20 (0/20 epochs)
[weighted_l1_dwt] epoch 005 | loss: 0.127872 | val SI-SDR: 18.9500 dB | best: 18.9500 dB | lr: 1.00e-03 | no improve: 0/20 (0/20 epochs)
[weighted_l1_dwt] epoch 006 | loss: 0.123005 | val SI-SDR: 19.2317 dB | best: 19.2317 dB | lr: 1.00e-03 | no improve: 0/20 (0/20 epochs)
[weighted_l1_dwt] epoch 007 | loss: 0.120006 | val SI-SDR: 19.3931 dB | best: 19.3931 dB | lr: 1.00e-03 | no improve: 0/20 (0/20 epochs)
[weighted_l1_dwt] epoch 008 | loss: 0.117